# Unsupervised Market Regime Detection — Bitcoin (BTC-USD)

**Goal:** Automatically identify distinct market regimes (bull, bear, crisis) in Bitcoin price history using unsupervised clustering — without any predefined labels.

**Approach:** Build financial features from raw OHLCV data, normalize them, and compare two clustering algorithms:
- **K-Means**: partition-based, identifies compact spherical clusters
- **DBSCAN**: density-based, naturally detects outliers (anomalous market events)

**Dataset:** BTC-USD daily data from Yahoo Finance (2018–2024)

## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

np.random.seed(42)
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
# Download Bitcoin daily data (2018–2024)
data = yf.download("BTC-USD", start="2018-01-01", end="2024-01-01", auto_adjust=False)
data = data[["Close", "Volume"]].dropna()

print(f"Dataset: {data.shape[0]} trading days")
data.head()

## 2. Feature Engineering — Financial Regime Descriptors

Each trading day is represented as a point in a multi-dimensional feature space.
Features are chosen to capture the key dimensions of market behavior:
- **Volatility**: how risky is the current environment?
- **Return**: is the market trending up or down?
- **Momentum**: what is the directional strength over the past month?
- **Volume**: how active is market participation?

Using a 30-day window captures medium-term regime dynamics rather than daily noise.

In [ ]:
df = data.copy()
df["return"] = df["Close"].pct_change()

# 30-day rolling volatility — risk level over past month
df["volatility_30d"]  = df["return"].rolling(30).std()

# 30-day cumulative return — medium-term trend direction
df["return_30d"]      = df["Close"].pct_change(30)

# 30-day momentum — absolute price displacement
df["momentum_30d"]    = df["Close"] - df["Close"].shift(30)

# Daily volume change — market activity spike detection
df["volume_change"]   = df["Volume"].pct_change()

df = df.dropna()
print(f"Dataset after feature construction: {df.shape[0]} days")
df[["return_30d", "volatility_30d", "momentum_30d", "volume_change"]].describe()

## 3. Raw Data Visualization — Geometric Intuition

Before clustering, visualize the data as a scatter plot.
Clusters must be visible geometrically before an algorithm can find them.

In [ ]:
plt.scatter(
    df["return_30d"],
    df["volatility_30d"],
    alpha=0.5, s=10
)
plt.title("Bitcoin Trading Days — 30d Return vs. 30d Volatility\nDenser region = calm market | Outliers = crisis or euphoria")
plt.xlabel("30-day Return")
plt.ylabel("30-day Volatility")
plt.grid(True)
plt.show()

## 4. Normalization — Critical for Distance-Based Clustering

K-Means and DBSCAN rely on **Euclidean distance**.
Without normalization, `momentum_30d` (expressed in USD, order of magnitude: thousands)
completely dominates the distance metric — other features become irrelevant.

StandardScaler brings all features to mean=0, std=1.

In [ ]:
features = ["return_30d", "volatility_30d", "momentum_30d", "volume_change"]
X = df[features]

scaler   = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features, index=df.index)

print("Before normalization:")
print(X.describe().loc[["mean", "std"]].round(4))
print()
print("After normalization:")
print(X_scaled.describe().loc[["mean", "std"]].round(4))

## 5. K-Means Clustering

K-Means partitions data into K clusters by minimizing within-cluster variance.
Each point is assigned to its nearest centroid.
We first determine the optimal K using two complementary methods.

In [ ]:
# --- Elbow method: inertia vs. K ---
K_range  = range(1, 11)
inertias = [KMeans(n_clusters=k, random_state=42).fit(X_scaled).inertia_ for k in K_range]

# --- Silhouette score: cluster separation quality ---
K_range_sil   = range(2, 11)
sil_scores    = [
    silhouette_score(X_scaled, KMeans(n_clusters=k, random_state=42).fit_predict(X_scaled))
    for k in K_range_sil
]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(list(K_range), inertias, marker='o')
axes[0].set_title("Elbow Method — Inertia vs. K\nLook for the inflection point")
axes[0].set_xlabel("Number of clusters K")
axes[0].set_ylabel("Inertia")
axes[0].grid(True)

axes[1].plot(list(K_range_sil), sil_scores, marker='o', color='orange')
axes[1].set_title("Silhouette Score vs. K\nHigher = better-separated clusters")
axes[1].set_xlabel("Number of clusters K")
axes[1].set_ylabel("Silhouette score")
axes[1].grid(True)

plt.tight_layout()
plt.show()

best_k = list(K_range_sil)[sil_scores.index(max(sil_scores))]
print(f"Best K by silhouette score: {best_k} (score = {max(sil_scores):.4f})")

In [ ]:
# Fit K-Means with K=3 — interpretable as calm / bull / bear+crisis
kmeans = KMeans(n_clusters=3, random_state=42)
df["cluster"] = kmeans.fit_predict(X_scaled)

# Visualize clusters in the return/volatility space
colors = {0: "steelblue", 1: "seagreen", 2: "tomato"}
for c in df["cluster"].unique():
    subset = df[df["cluster"] == c]
    plt.scatter(subset["return_30d"], subset["volatility_30d"],
                s=10, alpha=0.6, color=colors[c], label=f"Cluster {c}")

plt.title("K-Means Clusters — Bitcoin Market Regimes")
plt.xlabel("30-day Return")
plt.ylabel("30-day Volatility")
plt.legend()
plt.grid(True)
plt.show()

## 6. Financial Interpretation of Clusters

Cluster labels are just integers — financial meaning must be inferred from statistics.

In [ ]:
# Cluster statistics: mean and median of each feature
cluster_stats = df.groupby("cluster")[features].agg(["mean", "median"]).round(4)
print(cluster_stats)
print(f"\nCluster sizes:\n{df['cluster'].value_counts().sort_index()}")

In [ ]:
# Assign financial labels based on cluster statistics
# Adjust mapping after inspecting cluster_stats above
cluster_names = {
    0: "Calm Market",      # low volatility, neutral return
    1: "Bull Market",      # positive return, moderate volatility
    2: "Volatile / Bear"   # high volatility, negative or unstable return
}
df["regime"] = df["cluster"].map(cluster_names)

print("Regime distribution:")
print(df["regime"].value_counts())

In [ ]:
# Visualize regime transitions over the full price history
regime_colors = {"Calm Market": "steelblue", "Bull Market": "seagreen", "Volatile / Bear": "tomato"}

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for regime, color in regime_colors.items():
    subset = df[df["regime"] == regime]
    axes[0].scatter(subset.index, subset["Close"], s=5, alpha=0.6,
                    color=color, label=regime)
axes[0].set_title("BTC Price History Colored by Market Regime")
axes[0].set_ylabel("Price ($)")
axes[0].legend()
axes[0].grid(True)

for regime, color in regime_colors.items():
    subset = df[df["regime"] == regime]
    axes[1].scatter(subset.index, subset["volatility_30d"], s=5, alpha=0.6,
                    color=color, label=regime)
axes[1].set_title("30-day Volatility Colored by Regime")
axes[1].set_ylabel("Volatility")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. Impact of Normalization — Side-by-Side Comparison

Without normalization, `momentum_30d` (USD values, orders of magnitude larger)
dominates all distances and the clustering becomes meaningless.

In [ ]:
labels_no_scaling = KMeans(n_clusters=3, random_state=42).fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df["return_30d"], df["volatility_30d"],
                c=labels_no_scaling, cmap="viridis", alpha=0.6, s=10)
axes[0].set_title("K-Means WITHOUT Normalization\nClusters driven by momentum_30d scale only")
axes[0].set_xlabel("30-day Return")
axes[0].set_ylabel("30-day Volatility")
axes[0].grid(True)

axes[1].scatter(df["return_30d"], df["volatility_30d"],
                c=df["cluster"], cmap="viridis", alpha=0.6, s=10)
axes[1].set_title("K-Means WITH Normalization\nAll features contribute equally to distance")
axes[1].set_xlabel("30-day Return")
axes[1].set_ylabel("30-day Volatility")
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 8. DBSCAN — Density-Based Clustering & Anomaly Detection

Unlike K-Means, DBSCAN:
- Does **not** require specifying K in advance
- Finds clusters of **arbitrary shape** (not just spherical)
- Explicitly labels sparse/isolated points as **outliers** (label = -1)

In a financial context, outliers = anomalous market events (crashes, speculative peaks).

Key parameter: `eps` = neighborhood radius. Sensitivity is illustrated below.

In [ ]:
# Compare DBSCAN results across different eps values
eps_values = [0.2, 0.5, 1.0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, eps in zip(axes, eps_values):
    labels = DBSCAN(eps=eps, min_samples=5).fit_predict(X_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_pct  = (labels == -1).mean()

    ax.scatter(
        df["return_30d"], df["volatility_30d"],
        c=labels, cmap="plasma", alpha=0.6, s=10
    )
    ax.set_title(f"DBSCAN eps={eps}\n{n_clusters} clusters | {noise_pct:.1%} noise (label=-1)")
    ax.set_xlabel("30-day Return")
    ax.set_ylabel("30-day Volatility")
    ax.grid(True)

plt.suptitle("DBSCAN Sensitivity to eps Parameter", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Fit DBSCAN with the most interpretable eps
dbscan = DBSCAN(eps=0.5, min_samples=5)
df["dbscan_cluster"] = dbscan.fit_predict(X_scaled)

n_clusters = len(set(df["dbscan_cluster"].unique()) - {-1})
noise_pct  = (df["dbscan_cluster"] == -1).mean()

print(f"DBSCAN (eps=0.5): {n_clusters} clusters | {noise_pct:.1%} of days classified as outliers")
print()
print("Outlier days (anomalous events):")
outliers = df[df["dbscan_cluster"] == -1][["return_30d", "volatility_30d"]]
print(outliers.describe().round(4))

In [ ]:
# Highlight anomalous days on the price chart
outlier_dates = df[df["dbscan_cluster"] == -1].index

plt.figure(figsize=(14, 5))
plt.plot(df.index, df["Close"], color="steelblue", linewidth=0.8, label="BTC Price")
plt.scatter(outlier_dates, df.loc[outlier_dates, "Close"],
            color="tomato", s=15, zorder=5, label="DBSCAN outliers (anomalous days)")
plt.title("BTC Price — DBSCAN Outliers Highlighted\nRed dots = days with anomalous statistical behavior")
plt.xlabel("Date")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True)
plt.show()

## 9. K-Means vs. DBSCAN — Head-to-Head Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(
    df["return_30d"], df["volatility_30d"],
    c=df["cluster"], cmap="viridis", alpha=0.6, s=10
)
axes[0].set_title("K-Means (K=3)\nCompact spherical clusters, no outlier detection")
axes[0].set_xlabel("30-day Return")
axes[0].set_ylabel("30-day Volatility")
axes[0].grid(True)

axes[1].scatter(
    df["return_30d"], df["volatility_30d"],
    c=df["dbscan_cluster"], cmap="plasma", alpha=0.6, s=10
)
axes[1].set_title("DBSCAN (eps=0.5)\nArbitrary shapes + explicit outlier detection (dark = noise)")
axes[1].set_xlabel("30-day Return")
axes[1].set_ylabel("30-day Volatility")
axes[1].grid(True)

plt.tight_layout()
plt.show()

comparison = pd.DataFrame([
    {"Algorithm": "K-Means",  "Requires K": "Yes", "Outlier detection": "No",
     "Cluster shape": "Spherical", "Best for": "Well-separated regimes"},
    {"Algorithm": "DBSCAN",   "Requires K": "No",  "Outlier detection": "Yes",
     "Cluster shape": "Arbitrary", "Best for": "Anomaly detection"}
])
comparison.set_index("Algorithm")

## 10. Conclusion

**Key findings:**

- **Normalization is non-negotiable** for distance-based clustering — without it, `momentum_30d` dominates and clusters become meaningless
- **K-Means (K=3)** successfully identifies 3 interpretable market regimes: calm, bull, volatile/bear — transitions align with known Bitcoin market cycles (2018 bear, 2020–2021 bull, 2022 bear)
- **DBSCAN** detects anomalous trading days (outliers) corresponding to extreme events: COVID crash (March 2020), Terra/LUNA collapse (May 2022), FTX bankruptcy (November 2022)
- Clustering is **exploratory, not predictive** — regimes describe past behavior patterns; they do not guarantee future performance

**Practical applications:**
- Regime-conditional risk management (adjust position sizing based on detected regime)
- Anomaly detection for real-time risk alerts
- Feature for downstream ML models (regime label as an additional input)